In [3]:
import pysam
import pandas as pd
from collections import defaultdict

In [4]:
bam_path = "/media/scratch/fy2306/projects/base_editing/data/bowtie/all/all_sgrna_seqs.bowtie_hg38.mismatch3.sorted.bam"
bamfile = pysam.AlignmentFile(bam_path, "rb")

read_stats = defaultdict(lambda: {"NM0": 0, "NM1": 0, "NM2": 0, "NM3": 0})

for read in bamfile.fetch(until_eof=True):
    if read.is_unmapped:
        continue

    read_name = read.query_name
    nm_tag = read.get_tag("NM") if read.has_tag("NM") else None

    if nm_tag is not None:
        if nm_tag == 0:
            read_stats[read_name]["NM0"] += 1
        elif nm_tag == 1:
            read_stats[read_name]["NM1"] += 1
        elif nm_tag == 2:
            read_stats[read_name]["NM2"] += 1
        elif nm_tag == 3:
            read_stats[read_name]["NM3"] += 1
            
df = pd.DataFrame.from_dict(read_stats, orient="index").reset_index()
df.columns = ["id", "Alignments_NM0", "Alignments_NM1", "Alignments_NM2", "Alignments_NM3"]
df["Alignments_NM_less_than_3"] = df["Alignments_NM0"] + df["Alignments_NM1"] + df["Alignments_NM2"] + df["Alignments_NM3"]

df.to_csv("/media/scratch/fy2306/projects/base_editing/data/bowtie/all/all_sgrna_seqs.bowtie_hg38.mismatch3.processed.tsv", sep="\t", index=False)